# Online Retail II — cleaning & EDA

Prep notes for the Retail Executive Dashboard.

Combine both Excel years, drop junk lines, then check revenue / orders / customers against the Power BI cards (~£20M, ~40K orders, ~5.9K customers).


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data')
OUT = Path('outputs')
OUT.mkdir(parents=True, exist_ok=True)

r1 = pd.read_excel(DATA / 'online_retail_II.xlsx', sheet_name='Year 2009-2010')
r2 = pd.read_excel(DATA / 'online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([r1, r2], ignore_index=True).drop_duplicates()
df['Invoice'] = df['Invoice'].astype(str)
print('raw rows', len(df))
df.head()


In [ ]:
fee = ['POST', 'DOT', 'M', 'D', 'AMAZONFEE', 'BANK CHARGES', 'CRUK']
rev = df[
    (~df['Invoice'].str.startswith('C'))
    & (df['Quantity'] > 0)
    & (df['Price'] > 0)
    & (df['Description'].notna())
    & (~df['StockCode'].astype(str).isin(fee))
].copy()
rev['LineAmount'] = rev['Quantity'] * rev['Price']
print('revenue', round(rev['LineAmount'].sum(), 2))
print('orders', rev['Invoice'].nunique())
print('customers', rev['Customer ID'].nunique(dropna=True))
print('aov', round(rev['LineAmount'].sum() / rev['Invoice'].nunique(), 2))


In [ ]:
rev['InvoiceDate'] = pd.to_datetime(rev['InvoiceDate'])
monthly = rev.set_index('InvoiceDate').resample('MS')['LineAmount'].sum()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(monthly.index, monthly.values / 1e6, marker='o', markersize=3)
ax.set_title('Monthly merchandise revenue (£M)')
ax.set_ylabel('£M')
fig.tight_layout()
fig.savefig(OUT / 'monthly_revenue.png', dpi=120)
plt.show()

top_c = rev.groupby('Country')['LineAmount'].sum().sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(top_c.index[::-1], (top_c.values / 1e6)[::-1])
ax.set_title('Top countries by revenue (£M)')
fig.tight_layout()
fig.savefig(OUT / 'top_countries.png', dpi=120)
plt.show()
